# Spin Orbite model for 3d orbital
## Author: Mathieu Desmarais
### Date: 28-05-2026 

## Packages and modules

In [1]:
import numpy as np 
import qutip as qt                 # Package for quantum mecanic
import ufss as uf                  # Generation of doubled sided feynmann diagram
import matplotlib.pyplot as plt

# 2. La spectroscopie 2D
from qudpy.Classes import System   # Calcul and generation of 2D plot
import qudpy.plot_functions as pf

## Physics constant and definition of parameter

In [ ]:
hbar = 0.658211951 # hbar in eV*fs
T_SP_0= 14 # 14 Kelvu=in

E_d = 1
E_B = 1

## Random function definition

In [ ]:
##### Calcul of the dispersion for the spin-Peierls phases #####
def Spin_Peierls_dispersion(k, T, B, J, T_SP_0, delta_0, beta):
    alpha = 0.004
    T_SP = T_SP_0 * (1 - alpha * B**2)
    T_SP = max(0.0,T_SP)

    if T < T_SP: 
        delta = delta_0 * (1 - T/T_SP)**beta
    else: 
        delta=0

    #Cross-Fisher relation: the gap Delta scale like J* delta^(2/3)
    Delta = 2.0 * J * (delta **(2/3)) if delta > 0 else 0.0

    v = (np.pi * J) /2
    epsilon_k = np.sqrt(Delta**2 + (v * np.sin(k)**2))
    return epsilon_k


## Operator definition

Definition of the operator used to describe the Hamiltonians. We want to work in the k-space instead of the real space. By doing that, we are able to reduce considerably the calcul. The size of the system will not scale in power of the number of ions in the chains. 

## Hamiltonian Definition

For the minimal description, 3 Hamiltonian needed to be construct. We need:  
- Spin sector Hamiltonian

To simulate the spin sector, 


- Orbital sector Hamiltonian
- Spin-Orbite coupling Hamiltonian 

In [ ]:




def BuildKSpaceSpinHamiltonian(N_modes, T, B, J, T_SP_0, delta_0, beta, max_bosons=2):

    k_values = np.linspace(0, np.pi, N_modes)
    epsilon_k = [Spin_Peierls_dispersion(k,T,B,J,T_SP_0,delta_0,beta)for k in k_values]


    #Creating annihilation operator for each mode

    id_list = [qt.eye(max_bosons)] * N_modes
    annihilation_ops = []

    for i in range(N_modes):
        op_list = list(id_list)
        op_list[i] = qt.destroy(max_bosons)
        annihilation_ops.append(qt.tensor(op_list))

    H_spin = 0
    for i in range(N_modes):
        a=annihilation_ops[i]
        H_0 += epsilon_k[i] * a.dag() * a  

    return H_spin  



def BuildOrbitalHamiltonian(Delta_dark, Delta_bright):

    ket_g = qt.basis(3,0) # g> = [1 0 0]^T
    ket_d = qt.basis(3,1) # d> = [0 1 0]^T
    ket_b = qt.basis(3,2) # b - [0 0 1]^T

    #projection operator
    P_g = ket_g * ket_g.dag()
    P_d = ket_d * ket_d.dag()
    P_b = ket_b * ket_b.dag()

    H_orb = Delta_dark * P_d + Delta_bright * P_b

    dipole_op = ket_g* ket_b.dag() + ket_b * ket_g.dag()

    L_op = ket_d* ket_b.dag() + ket_b * ket_d.dag()

    return H_orb, dipole_op, L_op, ket_g, ket_d, ket_b 


def TotalHamiltonian(H_orb, dipole_op, L_op, H_spin_k, annihilations_ops, lambda_SOC):
    dim_orb = H_orb.dims[0]
    dim_spin = H_spin_k.dims[0]
          

## Dissipation (Lindblad Operator)

## Density Matrix 

## Impulsion sequence

## Feynmann Diagram

## Dynamic calcul

## Fourier Transform

## Signal extraction

## 2D spectra plot